In [16]:
pip install scikit-learn numpy matplotlib seaborn

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [17]:
"""
Titanic Data Analysis and JSON Export
Author: [Your Name]
Description: Analyze Titanic passenger data, engineer features, and export to JSON
"""
 
import pandas as pd
import numpy as np
import json
from pathlib import Path
 
# Set up paths
DATA_DIR = Path("data")
CSV_FILE = DATA_DIR / "titanic.csv"
JSON_FILE = DATA_DIR / "titanic_data.json"
 
# Create data directory if it doesn't exist
DATA_DIR.mkdir(exist_ok=True)
 
print("Project setup complete!")
print(f"Data directory: {DATA_DIR}")
print(f"CSV file location: {CSV_FILE}")

Project setup complete!
Data directory: data
CSV file location: data/titanic.csv


In [30]:
df = pd.read_csv(CSV_FILE)

In [31]:
print(f"Dataset loaded successfully! Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
print(df.head())

Dataset loaded successfully! Shape: (891, 12)

Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

First few rows:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 175

In [20]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [21]:
# Select numeric columns only
numeric_columns = df.select_dtypes(include=['number'])
# Calculate statistics (.mean, median, std)

In [22]:
# Show the column names
numeric_columns.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], dtype='object')

In [23]:
# Count missing values
print("\n" + "="*50)
print("MISSING VALUES ANALYSIS")
print("="*50)
missing_data = {}
for col in df.columns:
    missing_count = df[col].isna().sum()
    missing_percent = (missing_count / len(df)) * 100
    missing_data[col] = {
        "missing_count": missing_count,
        "missing_percent": missing_percent
    }
missing_df = pd.DataFrame(missing_data).T
missing_df.sort_values("missing_count", ascending=False)


MISSING VALUES ANALYSIS


,missing_count,missing_percent
Cabin,687.0,77.104377
Age,177.0,19.865320
Embarked,2.0,0.224467
PassengerId,0.0,0.000000
Name,0.0,0.000000
Pclass,0.0,0.000000
Survived,0.0,0.000000
Sex,0.0,0.000000
Parch,0.0,0.000000
SibSp,0.0,0.000000


In [24]:
# Create a copy of the dataframe for feature engineering
df_features = df.copy()
 
# Feature 1: Family Size
df_features['FamilySize'] = df_features['FamilySize'] = df_features['SibSp'] + df_features['Parch'] + 1
print(df_features[['SibSp', 'Parch', 'FamilySize']].head(10))

 
# Feature 2: Is Alone
df_features['IsAlone'] = df_features['FamilySize'] == 1
print(df_features[['FamilySize', 'IsAlone']].head(10))
 
# Feature 3: Age Groups
def categorize_age(age):
    """Categorize age into groups"""
    if pd.isna(age):
        return 'Unknown'
    elif age < 18:
        return 'Child'
    elif age < 30:
        return 'Young Adult'
    elif age < 50:
        return 'Adult'
    else:
        return 'Older Adult'

df_features['AgeGroup'] = df_features['Age'].apply(categorize_age)
print(df_features[['Age', 'AgeGroup']].head(10))

 
# Analyze feature differences between survivors and non-survivors
print("\n" + "="*50)
print("FEATURE ANALYSIS: SURVIVED vs NOT SURVIVED")
print("="*50)

print("\nFamily Size by Survival:")
family_survival = df_features.groupby('Survived')['FamilySize'].agg(['mean', 'median', 'std'])
print(family_survival)

print("\nIs Alone by Survival:")
alone_survival = df_features.groupby('Survived')['IsAlone'].mean()
print(alone_survival)

print("\nAge Group by Survival:")
agegroup_survival = pd.crosstab(df_features['AgeGroup'], df_features['Survived'], normalize='columns') * 100
print(agegroup_survival)
 
# Statistical test: Do these features help differentiate?
print("\n" + "="*50)
print("FEATURE DIFFERENTIATION ANALYSIS")
print("="*50)

survived = df_features[df_features['Survived'] == 1]
not_survived = df_features[df_features['Survived'] == 0]

print("\nFamily Size:")
print(f"  Survived mean: {survived['FamilySize'].mean():.2f}")
print(f"  Not Survived mean: {not_survived['FamilySize'].mean():.2f}")
print(f"  Difference: {abs(survived['FamilySize'].mean() - not_survived['FamilySize'].mean()):.2f}")

print("\nIs Alone:")
print(f"  Survived proportion alone: {survived['IsAlone'].mean():.2f}")
print(f"  Not Survived proportion alone: {not_survived['IsAlone'].mean():.2f}")
print(f"  Difference: {abs(survived['IsAlone'].mean() - not_survived['IsAlone'].mean()):.2f}")

print("\nAge Group Counts by Survival:")
print("Survived:")
print(survived['AgeGroup'].value_counts())
print("\nNot Survived:")
print(not_survived['AgeGroup'].value_counts())

   SibSp  Parch  FamilySize
0      1      0           2
1      1      0           2
2      0      0           1
3      1      0           2
4      0      0           1
5      0      0           1
6      0      0           1
7      3      1           5
8      0      2           3
9      1      0           2
   FamilySize  IsAlone
0           2    False
1           2    False
2           1     True
3           2    False
4           1     True
5           1     True
6           1     True
7           5    False
8           3    False
9           2    False
    Age     AgeGroup
0  22.0  Young Adult
1  38.0        Adult
2  26.0  Young Adult
3  35.0        Adult
4  35.0        Adult
5   NaN      Unknown
6  54.0  Older Adult
7   2.0        Child
8  27.0  Young Adult
9  14.0        Child

FEATURE ANALYSIS: SURVIVED vs NOT SURVIVED

Family Size by Survival:
              mean  median       std
Survived                            
0         1.883424     1.0  1.830669
1         1.938596     2.0 

Export the data

In [25]:
JSON_FILE

PosixPath('data/titanic_data.json')

In [26]:
import json
from datetime import datetime
import pandas as pd

# Step 6: Create Classes for JSON Export

class Passenger:
    """
    Represents a passenger with all their information.
    """
    def __init__(self, passenger_id, name, age, sex, survived, pclass,
                 fare, embarked=None, family_size=None, is_alone=None, age_group=None):
        
        self.passenger_id = int(passenger_id) if pd.notna(passenger_id) else None
        self.name = str(name) if pd.notna(name) else None
        self.age = float(age) if pd.notna(age) else None
        self.sex = str(sex) if pd.notna(sex) else None
        self.survived = int(survived) if pd.notna(survived) else None
        self.pclass = int(pclass) if pd.notna(pclass) else None
        self.fare = float(fare) if pd.notna(fare) else None
        self.embarked = str(embarked) if pd.notna(embarked) else None
        self.family_size = int(family_size) if pd.notna(family_size) else None
        self.is_alone = bool(is_alone) if pd.notna(is_alone) else None
        self.age_group = str(age_group) if pd.notna(age_group) else None

    def to_dict(self):
        """Convert passenger to dictionary for JSON serialization."""
        return {
            'passenger_id': self.passenger_id,
            'name': self.name,
            'age': self.age,
            'sex': self.sex,
            'survived': self.survived,
            'pclass': self.pclass,
            'fare': self.fare,
            'embarked': self.embarked,
            'family_size': self.family_size,
            'is_alone': self.is_alone,
            'age_group': self.age_group
        }


class TitanicDataset:
    """
    Represents the entire Titanic dataset with methods for JSON export.
    """
    def __init__(self, dataframe):
        self.dataframe = dataframe
        self.passengers = []
        self._create_passengers()

    def _create_passengers(self):
        """Create Passenger objects from dataframe."""
        for idx, row in self.dataframe.iterrows():
            passenger = Passenger(
                passenger_id=row.get('PassengerId', idx),
                name=row.get('Name', None),
                age=row.get('Age', None),
                sex=row.get('Sex', None),
                survived=row.get('Survived', None),
                pclass=row.get('Pclass', None),
                fare=row.get('Fare', None),
                embarked=row.get('Embarked', None),
                family_size=row.get('FamilySize', None),
                is_alone=row.get('IsAlone', None),
                age_group=row.get('AgeGroup', None)
            )
            self.passengers.append(passenger)

    def calculate_numeric_statistics(self):
        """Calculate mean, median, and std for numeric columns."""
        numeric_df = self.dataframe.select_dtypes(include=['number'])
        stats = {}

        for col in numeric_df.columns:
            stats[col] = {
                'mean': float(numeric_df[col].mean()) if pd.notna(numeric_df[col].mean()) else None,
                'median': float(numeric_df[col].median()) if pd.notna(numeric_df[col].median()) else None,
                'std': float(numeric_df[col].std()) if pd.notna(numeric_df[col].std()) else None
            }

        return stats

    def calculate_missing_values(self):
        """Calculate missing values for each column."""
        missing_data = {}

        for col in self.dataframe.columns:
            missing_count = int(self.dataframe[col].isna().sum())
            missing_percent = float((missing_count / len(self.dataframe)) * 100)

            missing_data[col] = {
                'missing_count': missing_count,
                'missing_percent': missing_percent
            }

        return missing_data

    def get_summary_stats(self):
        """Get summary statistics."""
        survived_count = sum(1 for p in self.passengers if p.survived == 1)
        not_survived_count = sum(1 for p in self.passengers if p.survived == 0)

        ages = [p.age for p in self.passengers if p.age is not None]
        fares = [p.fare for p in self.passengers if p.fare is not None]

        return {
            'total_passengers': len(self.passengers),
            'survived': survived_count,
            'did_not_survive': not_survived_count,
            'average_age': round(sum(ages) / len(ages), 2) if ages else None,
            'average_fare': round(sum(fares) / len(fares), 2) if fares else None
        }

    def to_json(self, filename='titanic_data.json'):
        """Export dataset to JSON file."""
        data = {
            'metadata': {
                'dataset_name': 'Titanic Passenger Dataset',
                'export_date': datetime.now().isoformat(),
                'total_passengers': len(self.passengers),
                'survival_rate': float(self.dataframe['Survived'].mean()),
                'columns': list(self.dataframe.columns)
            },
            'summary_statistics': self.get_summary_stats(),
            'numeric_statistics': self.calculate_numeric_statistics(),
            'missing_values': self.calculate_missing_values(),
            'passengers': [p.to_dict() for p in self.passengers]
        }

        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2)

        print(f"Data exported to {filename}")
        return data


# Create dataset object and export
if 'df_features' in locals() and not df_features.empty:
    dataset = TitanicDataset(df_features)

    print("\n" + "=" * 50)
    print("DATASET OBJECT CREATED")
    print("=" * 50)
    print(f"Number of passenger objects: {len(dataset.passengers)}")

    print("\nSummary statistics:")
    summary = dataset.get_summary_stats()
    for key, value in summary.items():
        print(f"{key}: {value}")

    # Export to JSON
    dataset.to_json('titanic_data.json')


DATASET OBJECT CREATED
Number of passenger objects: 891

Summary statistics:
total_passengers: 891
survived: 342
did_not_survive: 549
average_age: 29.7
average_fare: 32.2
Data exported to titanic_data.json


Step 7

In [27]:
JSON_FILE

PosixPath('data/titanic_data.json')

In [28]:
# Additional validation: Load and inspect JSON
with open(JSON_FILE, 'r', encoding='utf-8') as f:
    json_data = json.load(f)
 
# Print summary of JSON data and verify content

In [29]:
#from sklearn.preprocessing import PowerTransformer
#power_transformer = PowerTransformer()